In [ ]:
from sklearn.model_selection import train_test_split
import os

### 1. Remove too big/small files and make data splits

In [ ]:
wav_folder = '/home/sandra/projects/matcha/data/peeter/peeter_wav_resampled'
txt_folder = '/home/sandra/projects/matcha/data/peeter/peeter_txt'

# Output folder for files in format mentoned above
training_files_folder = '/home/sandra/projects/matcha/data/peeter/peeter_training_sets/exp-2'

In [ ]:
sample_file_selection = [] # Files from the folder that are in certain size range
files_discarded = 0
files_included = 0

# Filtering out files that are too small or too large
for file in os.listdir(wav_folder):
    fsize = os.path.getsize(os.path.join(wav_folder, file))
    if fsize < 44000 or fsize > 1000000:
        files_discarded += 1 
        continue
    files_included += 1
    sample_file_selection.append(file)

print(f'{files_included} files included in datasets. {files_discarded} files discarded.')

4979 files included in datasets. 338 files discarded.


In [ ]:
# Splitting into train, validation and test sets

train, test_val = train_test_split(sample_file_selection, train_size=0.95, shuffle=True, random_state=5)

val, test = train_test_split(test_val, train_size=0.2, shuffle=True, random_state=5)

In [5]:
len(train), len(test), len(val)

(4730, 200, 49)

In [6]:
train[:15]

['lause04578.wav',
 'lause02959.wav',
 'lause03662.wav',
 'lause01729.wav',
 'lause01619.wav',
 'lause00984.wav',
 'lause03098.wav',
 'lause00874.wav',
 'lause05152.wav',
 'lause02889.wav',
 'lause00094.wav',
 'lause02018.wav',
 'lause02458.wav',
 'lause03566.wav',
 'lause03438.wav']

### 2. Processing data to correct format

Input files must adhere to the following format, where DUMMY marks the wav-file path of the sentence that follows. 


    DUMMY/LJ049-0022.wav|The Secret Service believed that it was very doubtful that any President would ride regularly in a vehicle with a fixed top, even though transparent.
    DUMMY/LJ033-0042.wav|Between the hours of eight and nine p.m. they were occupied with the children in the bedrooms located at the extreme east end of the house.
    DUMMY/LJ016-0117.wav|The prisoner had nothing to deal with but wooden panels, and by dint of cutting and chopping he got both the lower panels out.


In [ ]:
# Outout files for storing train, validation and test data in correct format for Matcha-TTS
train_filelist_fname = 'est_audio_text_train_filelist_4400_1000000.txt'
validation_filelist_fname = 'est_audio_text_val_filelist_4400_10000000.txt'
test_filelist_fname = 'est_audio_text_test_filelist_4400_10000000.txt'

In [ ]:
# Functions to combine information into correct format

def read_sentence_from_file(sentence: str, txt_folder):
    if not sentence.endswith('.txt'):
        sentence = sentence + '.txt'
    with open(os.path.join(txt_folder, sentence)) as f:
        sentence_text = f.read() + '\n'
    return sentence_text
        

def write_to_filelist(filelist_name, wav_file_subset, wav_file_reference_path, txt_folder):
    lines = []
    for fname in wav_file_subset:
        sentence_id = fname.split('.')[0]
        wav_reference = f'{wav_file_reference_path}/{fname}'
        sentence = read_sentence_from_file(sentence_id, txt_folder)
        lines.append(f'{wav_reference}|{sentence}')
    
    with open(filelist_name, 'w') as f:
        f.writelines(lines)


In [ ]:
# Function calls for processing train, val and test sets

# Process train data
write_to_filelist(f'{training_files_folder}/{train_filelist_fname}', train, wav_folder, txt_folder)

# Process validation data
write_to_filelist(f'{training_files_folder}/{validation_filelist_fname}', val, wav_folder, txt_folder)

# Process test data
write_to_filelist(f'{training_files_folder}/{test_filelist_fname}', test, wav_folder, txt_folder)